# Adsorption energy cycle with gadivasp
Charged Ni(111) slab (+1 e): relax slab, relax NH3 on slab, gas-phase NH3, then E_ads. Requires Gadi; the analysis cells at the end run anywhere with your own `energies`.

In [ ]:
import gadivasp as gv
from ase.build import add_adsorbate, molecule
from gadivasp.structures import ni111_slab, fix_bottom_layers, gas_molecule

cfg = gv.load_config()

## 1. Build and submit

In [ ]:
slab = fix_bottom_layers(ni111_slab(size=(4, 4, 5), total_height=18.0), n_free_layers=3)

ads = slab.copy()
nh3 = molecule('NH3')
add_adsorbate(ads, nh3, height=2.0, position='ontop')

calcs = {
    'slab':  gv.Calculation('slab_1e',  slab, cfg, preset='relax', charge=+1),
    'nh3_slab': gv.Calculation('nh3_slab_1e', ads, cfg, preset='relax', charge=+1),
    'nh3_gas': gv.Calculation('nh3_gas', gas_molecule('NH3'), cfg, preset='molecule',
                              executable='vasp_gam', resources=dict(ncpus=28, mem='30gb', walltime='1:00:00')),
}
for c in calcs.values():
    c.prepare(); c.submit()
gv.wait_for([c.job_id for c in calcs.values()])

## 2. Adsorption energy

In [ ]:
e = {k: c.energy() for k, c in calcs.items()}
e_ads = gv.adsorption_energy(e['nh3_slab'], e['slab'], e['nh3_gas'])
print(f'E_ads(NH3, +1e) = {e_ads:.3f} eV')

## 3. Vacuum convergence of a charged slab
Fit E(L) = a/L² + b/L³ + c and use c as the infinite-vacuum energy.

In [ ]:
import matplotlib.pyplot as plt
from gadivasp.plotting import vacuum_fit_plot

heights  = [18.0, 20.5, 23.0, 25.5, 28.0]
energies = [c.energy() for c in vacuum_series]   # fill in from your vacuum-sweep Calculations
fit = gv.fit_vacuum(heights, energies)
vacuum_fit_plot(heights, energies, fit); plt.show()